# Count of Gates for Shift and Phase Oracle in $L \times L$ grid 

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit.visualization import *
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.quantum_info import Statevector
from numpy import pi
import numpy as np
from matplotlib import pyplot as plt

# Aer is now a separate package (qiskit-aer)
from qiskit_aer import AerSimulator

In [2]:
Q = 8

In [3]:

N = Q*Q # Total Number of vertex in the grid
l = 4/N # Valule for self loop

In [4]:
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)

In [5]:
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")

In [6]:
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)

In [7]:
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')

In [8]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_4: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_5: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [9]:
def superposition(circuit, Q):
    num_states = int(2*np.log2(Q))
    for i in range(0,num_states):
        circuit.h(i)

In [10]:
one_step.append(coin_prep, coin)

In [11]:
def shift(circuit, Q):
    num_states = 3 + int(2*np.log2(Q))
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    circuit.x(num_states-1)
    E = int(np.log2(Q))
    D = E
    for i in range(int(np.log2(Q))):
        x = list(range(0,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(0,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    E = 2*int(np.log2(Q))
    for i in range(int(np.log2(Q))):
        x = list(range(D,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(D,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-1)
    circuit.x(num_states-1)
    circuit.mcx([num_states-1],num_states-3)
    circuit.x(num_states-1)

# 8x8

Initialization

In [13]:
x = 17 #number of steps
Q = 8
l = 4/(Q*Q)

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 11072), ('t', 6850), ('tdg', 6834), ('cx', 2)])

# Phase Oracle 

In [14]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.decompose().decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 474964),
             ('t', 289768),
             ('tdg', 289757),
             ('cx', 84),
             ('x', 3)])

# For one shift

In [15]:
Q = 8
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)

In [16]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)

In [17]:
one.count_ops()

OrderedDict([('cx', 289), ('t', 223), ('tdg', 205), ('h', 120), ('x', 8)])

# For One Coin

In [18]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 281022),
             ('t', 173781),
             ('tdg', 173758),
             ('cx', 19),
             ('x', 2),
             ('s', 2)])

# 16x16

In [19]:
Q = 16
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')


one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()


OrderedDict([('h', 10520), ('t', 6429), ('tdg', 6413), ('cx', 2)])

# Phase Oracle

In [20]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
# phase_circuit.draw()
phase_circuit.decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 1465404),
             ('t', 896187),
             ('tdg', 896173),
             ('cx', 192),
             ('x', 22),
             ('z', 1)])

# For one Shift Oporation

In [21]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('cx', 457), ('t', 353), ('tdg', 329), ('h', 200), ('x', 8)])

# Coin

In [22]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 292225),
             ('t', 179878),
             ('tdg', 179856),
             ('cx', 19),
             ('x', 2),
             ('s', 2)])

# 32x32

In [23]:
Q = 32
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()


OrderedDict([('h', 10616), ('t', 6531), ('tdg', 6515), ('cx', 2)])

# phase oracle 

In [24]:

A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 2407588),
             ('t', 1472941),
             ('tdg', 1472921),
             ('cx', 344),
             ('x', 21),
             ('z', 1)])

# For One step Shift

In [25]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()
one = transpile(one_step, basis_gates=["cx","rz","h","t","tdg","x","crz"],optimization_level=0)
one.count_ops()

OrderedDict([('cx', 573),
             ('t', 336),
             ('tdg', 310),
             ('h', 288),
             ('x', 42),
             ('rz', 30)])

# Coin

In [26]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 287653),
             ('t', 177048),
             ('tdg', 177016),
             ('cx', 19),
             ('x', 2),
             ('s', 2)])

# 64x64

In [27]:
Q = 64
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_prep, coin)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()


OrderedDict([('h', 10860), ('t', 6713), ('tdg', 6697), ('cx', 2)])

# Phase Oracle

In [28]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.decompose().count_ops()
one = transpile(phase_circuit, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 2700458),
             ('t', 1652793),
             ('tdg', 1652771),
             ('cx', 576),
             ('x', 34),
             ('z', 1)])

# For one step Shift

In [29]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one_step.decompose().count_ops()
one = transpile(one_step, basis_gates=["cx","rz","h","t","tdg","x","crz"],optimization_level=0)
one.count_ops()

OrderedDict([('cx', 773),
             ('t', 456),
             ('tdg', 424),
             ('h', 396),
             ('x', 58),
             ('rz', 30)])

# Coin

In [30]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)
#one_step.decompose().decompose().decompose().count_ops()
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0)
one.count_ops()

OrderedDict([('h', 286328),
             ('tdg', 175705),
             ('t', 175700),
             ('cx', 19),
             ('x', 2),
             ('s', 2)])

# VERSION

In [43]:
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit.visualization import *
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.quantum_info import Statevector
from numpy import pi
import numpy as np
from matplotlib import pyplot as plt
from qiskit_aer import AerSimulator

# ==========================================
# 1. Define Circuit & Transpilation Settings
# ==========================================
opt_level = 3
transpile_seed = 42

# Decomposition methods for MCX/MCZ in Qiskit are set during circuit building.
# Common modes: 'noancilla', 'recursion', 'v-chain', 'v-chain-dirty'
mcx_decomp_mode = 'noancilla' 
ancillas_permitted = "None" # Would be "Clean" for 'v-chain' or "Dirty" for 'v-chain-dirty'

# Create a sample circuit to demonstrate
qc = QuantumCircuit(4)
qc.h(0)
# Multi-controlled X gate using the specified decomposition mode
qc.mcx(control_qubits=[0, 1, 2], target_qubit=3, mode=mcx_decomp_mode)
qc.measure_all()

# ==========================================
# 2. Setup Backend & Transpile
# ==========================================
backend = AerSimulator()

# Extract hardware/simulator constraints
basis_gates = backend.configuration().basis_gates
coupling_map = backend.configuration().coupling_map

# Run the transpiler
transpiled_qc = transpile(
    qc, 
    backend=backend, 
    basis_gates=basis_gates,
    optimization_level=opt_level, 
    seed_transpiler=transpile_seed
)

# ==========================================
# 3. Print the Requested Metadata
# ==========================================
print("--- Versions ---")
print(f"Qiskit version:               {qiskit.__version__}")
print(f"Simulator/Compiler version:   {qiskit_aer.__version__} (qiskit-aer)")

print("\n--- Transpiler & Hardware Configuration ---")
print(f"Basis gate set:               {basis_gates}")
print(f"Coupling map, if any:         {coupling_map}")
print(f"Transpiler opt level:         {opt_level}")
print(f"Seed for transpilation:       {transpile_seed}")

print("\n--- Circuit Specifics ---")
print(f"Multi-controlled X/Z decomp:  {mcx_decomp_mode} (set via `mode` arg in circuit.mcx)")
print(f"Clean/dirty ancillas allowed: {ancillas_permitted}")
print(f"Global phase:                 {transpiled_qc.global_phase} (Tracked by Qiskit, usually ignored by physical hardware unless doing phase estimation)")

print("\n--- Metrics & Reporting Context ---")
# Depth
print(f"Reported depth value:         {transpiled_qc.depth()}")
print("Is depth logical or hardware? LOGICAL DEPENDENCY DEPTH.")
print("                              (Hardware scheduled depth requires running a Scheduling pass like ASAP/ALAP.)")

# Counts context
print("Counts status:                AFTER hardware routing.")
print("                              (Because execution happens on the `transpiled_qc` which has already been routed to the backend's coupling map.)")

C:\Users\nisha\AppData\Local\Temp\ipykernel_1756\2814249214.py:28: DeprecationWarning: ``qiskit.circuit.quantumcircuit.QuantumCircuit.mcx()``'s argument ``mode`` is deprecated as of Qiskit 2.1. It will be removed no earlier than 3 months after the release date. Instead, add a generic MCXGate to the circuit and specify the synthesis method via the ``hls_config`` in the transpilation. Alternatively, specific decompositions are available at https://qisk.it/mcx.
  qc.mcx(control_qubits=[0, 1, 2], target_qubit=3, mode=mcx_decomp_mode)
c:\Users\nisha\AppData\Local\Programs\Python\Python314\Lib\site-packages\qiskit\compiler\transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


--- Versions ---
Qiskit version:               2.3.1
Simulator/Compiler version:   0.17.2 (qiskit-aer)

--- Transpiler & Hardware Configuration ---
Basis gate set:               ['ccx', 'ccz', 'cp', 'crx', 'cry', 'crz', 'cswap', 'csx', 'cu', 'cu1', 'cu2', 'cu3', 'cx', 'cy', 'cz', 'diagonal', 'ecr', 'h', 'id', 'mcp', 'mcphase', 'mcr', 'mcrx', 'mcry', 'mcrz', 'mcswap', 'mcsx', 'mcu', 'mcu1', 'mcu2', 'mcu3', 'mcx', 'mcx_gray', 'mcy', 'mcz', 'multiplexer', 'p', 'pauli', 'r', 'roerror', 'rx', 'rxx', 'ry', 'ryy', 'rz', 'rzx', 'rzz', 's', 'sdg', 'store', 'swap', 'sx', 'sxdg', 't', 'tdg', 'u', 'u1', 'u2', 'u3', 'unitary', 'x', 'y', 'z', 'break_loop', 'continue_loop', 'delay', 'for_loop', 'if_else', 'initialize', 'kraus', 'qerror_loc', 'quantum_channel', 'reset', 'roerror', 'save_amplitudes', 'save_amplitudes_sq', 'save_clifford', 'save_density_matrix', 'save_expval', 'save_expval_var', 'save_matrix_product_state', 'save_probabilities', 'save_probabilities_dict', 'save_stabilizer', 'save_state'